In [1]:
 # Imports
import pandas as pd
import numpy as np
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import pickle

# Load
movies = pd.read_csv('../data/processed/movies.csv')
ratings = pd.read_csv('../data/processed/merged.csv')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/purshotam_kumar/nltk_data...


In [2]:
# Simulate sentiment from ratings
# We don't have review text in ML-1M, so we derive a sentiment proxy from ratings
# Rating >= 4 = positive, <= 2 = negative, 3 = neutral
def rating_to_sentiment(rating):
    if rating >= 4:
        return 1.0
    elif rating <= 2:
        return -1.0
    else:
        return 0.0

ratings['sentiment'] = ratings['rating'].apply(rating_to_sentiment)

In [3]:
# Aggregate sentiment per movie
movie_sentiment = ratings.groupby('movieId').agg(
    avg_sentiment=('sentiment', 'mean'),
    review_count=('rating', 'count')
).reset_index()

# Confidence weight — more reviews = more reliable
movie_sentiment['weighted_sentiment'] = (
    movie_sentiment['avg_sentiment'] * 
    np.log1p(movie_sentiment['review_count'])
)

# Normalize to 0-1
min_s = movie_sentiment['weighted_sentiment'].min()
max_s = movie_sentiment['weighted_sentiment'].max()
movie_sentiment['sentiment_score'] = (
    (movie_sentiment['weighted_sentiment'] - min_s) / (max_s - min_s)
)

print(movie_sentiment.head())

   movieId  avg_sentiment  review_count  weighted_sentiment  sentiment_score
0        1       0.759750          2077            5.803850         0.897468
1        2       0.192582           701            1.262170         0.497494
2        3       0.018828           478            0.116204         0.396571
3        4      -0.182353           170           -0.937597         0.303766
4        5       0.047297           296            0.269298         0.410054


In [4]:
# Reranker function
def sentiment_rerank(recommendations_df, alpha=0.3):
    """
    recommendations_df: output from ensemble_recommend (has 'title','genres','score')
    alpha: how much sentiment influences final score (0=ignore, 1=only sentiment)
    """
    merged = recommendations_df.merge(
        movies[['movieId', 'title']], on='title', how='left'
    ).merge(
        movie_sentiment[['movieId', 'sentiment_score']], on='movieId', how='left'
    )
    
    merged['sentiment_score'] = merged['sentiment_score'].fillna(0.5)
    merged['final_score'] = (
        (1 - alpha) * merged['score'] + alpha * merged['sentiment_score']
    )
    
    return merged[['title', 'genres', 'score', 'sentiment_score', 'final_score']]\
           .sort_values('final_score', ascending=False)

In [5]:
# Test (run ensemble first then rerank)
# Paste ensemble output here as a test
sample = pd.DataFrame({
    'title': ['Toy Story 2 (1999)', "Bug's Life, A (1998)", 'Chicken Run (2000)',
              'Iron Giant, The (1999)', 'Mulan (1998)'],
    'genres': ['Animation|Children\'s|Comedy'] * 5,
    'score': [0.936, 0.917, 0.911, 0.896, 0.878]
})

print(sentiment_rerank(sample))

# Cell 7 — Save sentiment data
movie_sentiment.to_csv('../data/processed/movie_sentiment.csv', index=False)
pickle.dump(movie_sentiment, open('../models/sentiment.pkl', 'wb'))
print("Saved.")

                    title                       genres  score  \
0      Toy Story 2 (1999)  Animation|Children's|Comedy  0.936   
1    Bug's Life, A (1998)  Animation|Children's|Comedy  0.917   
2      Chicken Run (2000)  Animation|Children's|Comedy  0.911   
3  Iron Giant, The (1999)  Animation|Children's|Comedy  0.896   
4            Mulan (1998)  Animation|Children's|Comedy  0.878   

   sentiment_score  final_score  
0         0.891181     0.922554  
1         0.804983     0.883395  
2         0.785768     0.873430  
3         0.788122     0.863637  
4         0.693715     0.822714  
Saved.
